# Volume 1. Lexicon And Readout

Question: after several assemblies are formed in one area, can a new probe be read back as the closest known label?

This notebook keeps the story concrete but modest. Labels such as `red`, `blue`, and `green` are handles for trained sparse patterns. Readout is nearest-neighbor overlap against a stored lexicon, not semantics.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from neural_assemblies.assembly_calculus import (
    Assembly,
    build_lexicon,
    chance_overlap,
    fuzzy_readout,
    readout_all,
)
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import plot_assemblies, plot_overlap_matrix


The cast is one lexical area and three stimulus labels. Keeping all references in the same area makes the overlap matrix meaningful.


In [ ]:
N = 4_000
K = 60
BETA = 0.08
ROUNDS = 8
WORDS = ["red", "blue", "green"]

brain = Brain(p=0.05, save_winners=True, seed=7, engine="numpy_sparse")
stimuli = {word: f"stim_{word}" for word in WORDS}
for stimulus in stimuli.values():
    brain.add_stimulus(stimulus, K)
brain.add_area("LEX", N, K, BETA)

pd.DataFrame({
    "word": WORDS,
    "stimulus": [stimuli[word] for word in WORDS],
    "area": "LEX",
    "winners_per_assembly": K,
    "density": K / N,
})


`build_lexicon` forms one reference assembly per word. The table gives a quick audit trail: area, density, and a few winner indices.


In [ ]:
lexicon = build_lexicon(brain, "LEX", WORDS, stimuli, rounds=ROUNDS)

pd.DataFrame([
    {
        "word": word,
        "area": assembly.area,
        "winner_count": len(assembly),
        "density": len(assembly) / N,
        "sample_winners": assembly.winners[:10].tolist(),
    }
    for word, assembly in lexicon.items()
])


In [ ]:
fig, axes = plot_assemblies(
    [lexicon[word] for word in WORDS],
    N,
    titles=[f"LEX: {word}" for word in WORDS],
    subtitles=[f"{K} winners | density {K / N:.3f}" for _ in WORDS],
    colors=["#e15554", "#4d9de0", "#59a14f"],
    marker_size=18,
)
plt.show()
plt.close(fig)


Same-area overlap should have a clear diagonal. The expected random baseline is `K / N`, which is low for this setup.


In [ ]:
ax, matrix = plot_overlap_matrix(
    [lexicon[word] for word in WORDS],
    labels=WORDS,
    cmap="viridis",
)
ax.set_title(f"LEX reference overlap (chance ~= {chance_overlap(K, N):.3f})")
plt.show()
plt.close(ax.figure)


A probe can be imperfect and still read out correctly if it is closest to the intended reference. Here we replace a small fraction of the `red` winners with fresh neuron IDs.


In [ ]:
rng = np.random.default_rng(123)
reference = lexicon["red"]
replacements = 12
keep = np.array(reference.winners[replacements:], dtype=np.uint32)
pool = np.setdiff1d(np.arange(N, dtype=np.uint32), reference.winners)
injected = rng.choice(pool, size=replacements, replace=False).astype(np.uint32)
noisy_red_probe = Assembly("LEX", np.concatenate([keep, injected]))

readout_scores = readout_all(noisy_red_probe, lexicon)
pd.DataFrame(readout_scores, columns=["word", "overlap_with_probe"])


In [ ]:
thresholds = [0.25, 0.5, 0.7, 0.9]
pd.DataFrame({
    "threshold": thresholds,
    "readout": [fuzzy_readout(noisy_red_probe, lexicon, threshold=threshold) for threshold in thresholds],
})


In [ ]:
scores = pd.DataFrame(readout_scores, columns=["word", "overlap"])
fig, ax = plt.subplots(figsize=(5.5, 3.2))
bars = ax.bar(scores["word"], scores["overlap"], color=["#e15554", "#4d9de0", "#59a14f"])
ax.axhline(chance_overlap(K, N), color="#555555", linestyle="--", label="chance baseline")
ax.bar_label(bars, labels=[f"{value:.2f}" for value in scores["overlap"]], padding=3)
ax.set_ylim(0, 1.05)
ax.set_ylabel("overlap with noisy probe")
ax.set_title("Readout scores for a damaged red probe")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()
plt.close(fig)


Readout is therefore an inspectable decision rule: store reference assemblies, measure overlaps, and choose the best label only if the overlap clears a threshold. Later notebooks can build richer labels on top of this, but the primitive is just sparse-set comparison.
